# Create the wave metamodel

Pipeline (settings from ``NorthCarolina.json`` → ``metamodel`` / ``variables``).

1. **Train / load** models (`fit_models`) — MDA + PCA + GP per goal from SnapWave static cases.
2. **Bulk reconstruction** (`reconstruct_bulk`) — canonical `hs` / `tp` / `wd` / `ds`.
3. **Zero-fill** missing forcing (`apply_missing_forcing_zeros`).
4. **Spectral reconstruction** (`reconstruct_spectral`) — wavespectra stats `hs` / `tp` / `tm01` / `dm`.
5. **Save** (`save_spectral`) → `outputs/spectral_metamodel.nc`.

In [1]:
from utils import metamodel as mm
from utils.config import GOALS, METAMODEL, reload

reload()  # pick up edits to NorthCarolina.json without restarting kernel
print(f"goals            : {len(GOALS)}")
print(f"MDA points       : {METAMODEL['num_mda_points']}")
print(f"spectral output  : {mm.spectral_output_path()}")

goals            : 14
MDA points       : 450
spectral output  : /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc


## Parameters

Tweak these while experimenting — especially `time` and `stations` for the
spectral step.

In [2]:
# --- training --------------------------------------------------------------
force_retrain = False  # set True to overwrite existing .pkl models

# --- bulk reconstruction ---------------------------------------------------
time_bulk = "1986-01"  # any xarray time selector (month, day, slice, ...)

# --- spectral reconstruction (bottleneck) ------------------------------------
time_spec = None  # None → use full bulk window; or e.g. "1986-01-15"
sites = None  # flexible: int, slice, list, "buoy", "isobath", name
# sites = slice(0, 3)   # quick smoke test on first 3 sites
# sites = -1              # single site by index

## 1 · Train / load models

In [3]:
models = mm.fit_models(force_retrain=force_retrain)
len(models["mda"]), len(models["pca"]), len(models["gp"])

2026-06-17 16:08:01,384 - NorthCarolina.metamodel - INFO - Fitting models for 14 goal(s)  (num_mda_points=450, force_retrain=False)
2026-06-17 16:08:01,595 - NorthCarolina.metamodel - INFO - Goal  1: loaded existing models
2026-06-17 16:08:01,632 - NorthCarolina.metamodel - INFO - Goal  2: loaded existing models
2026-06-17 16:08:01,669 - NorthCarolina.metamodel - INFO - Goal  3: loaded existing models
2026-06-17 16:08:01,701 - NorthCarolina.metamodel - INFO - Goal  4: loaded existing models
2026-06-17 16:08:01,734 - NorthCarolina.metamodel - INFO - Goal  5: loaded existing models
2026-06-17 16:08:01,768 - NorthCarolina.metamodel - INFO - Goal  6: loaded existing models
2026-06-17 16:08:01,799 - NorthCarolina.metamodel - INFO - Goal  7: loaded existing models
2026-06-17 16:08:01,835 - NorthCarolina.metamodel - INFO - Goal  8: loaded existing models
2026-06-17 16:08:01,973 - NorthCarolina.metamodel - INFO - Goal  9: loaded existing models
2026-06-17 16:08:02,006 - NorthCarolina.metamodel

(14, 14, 14)

## 2 · Bulk reconstruction

In [4]:
bulk = mm.reconstruct_bulk(time_bulk, models=models)
print(f"size : {bulk.nbytes * 1e-9:.2f} GB")
bulk

2026-06-17 16:08:02,631 - NorthCarolina.metamodel - INFO - Bulk reconstruction: time='1986-01', goals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], partitions=['bulk', 'sea', 'swell1', 'swell2', 'swell3']
2026-06-17 16:08:02,639 - NorthCarolina.metamodel - INFO - Goal  1 / bulk   : reconstructing 744 / 744 timesteps
2026-06-17 16:08:03,159 - NorthCarolina.metamodel - INFO - Goal  2 / bulk   : reconstructing 744 / 744 timesteps
INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed max custom scaler for ds is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_u is bigger than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_u is lower than datapoint
INFO:ExactGPInterpolation:Proposed min custom scaler for wd_v is bigger than datapoin

size : 0.36 GB


<xarray.Dataset> Size: 360MB
Dimensions:      (time: 744, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 90MB 0.4938 ... 9.333...
    tp           (goal, partition, time, sites) float64 90MB 4.805 ... 7.528
    wd           (goal, partition, time, sites) float64 90MB 174.2 ... 65.59
    ds           (goal, partition, time, sites) float64 90MB 37.0 37.0 ... 9.8

## 2b · Zero-fill missing forcing

Sea/swell partitions can be NaN when a component is absent. The metamodel only
predicts valid timesteps; this step sets those gaps to ``0.0`` before the
spectral sum.

In [5]:
bulk = mm.apply_missing_forcing_zeros(bulk)
bulk

2026-06-17 16:08:40,646 - NorthCarolina.metamodel - INFO - Zero-filled 11266560 NaN values in hs, tp, wd, ds


<xarray.Dataset> Size: 360MB
Dimensions:      (time: 744, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 90MB 0.4938 ... 9.333...
    tp           (goal, partition, time, sites) float64 90MB 4.805 ... 7.528
    wd           (goal, partition, time, sites) float64 90MB 174.2 ... 65.59
    ds           (goal, partition, time, sites) float64 90MB 37.0 37.0 ... 9.8

## 3 · Spectral reconstruction

Start with a **small** `sites` selection above, then scale up.

In [6]:
spectral = mm.reconstruct_spectral(bulk, sites=sites, time=time_spec)
spectral

2026-06-17 16:08:40,723 - NorthCarolina.metamodel - INFO - Spectral reconstruction: 216 site(s), partitions=['sea', 'swell1', 'swell2', 'swell3'], time=None
2026-06-17 16:08:40,724 - NorthCarolina.metamodel - INFO - Spectral site   1 / 216: 5m_10
2026-06-17 16:08:41,665 - NorthCarolina.metamodel - INFO - Spectral site   2 / 216: 5m_11
2026-06-17 16:08:42,559 - NorthCarolina.metamodel - INFO - Spectral site   3 / 216: 5m_12
2026-06-17 16:08:43,434 - NorthCarolina.metamodel - INFO - Spectral site   4 / 216: 5m_13
2026-06-17 16:08:44,288 - NorthCarolina.metamodel - INFO - Spectral site   5 / 216: 5m_14
2026-06-17 16:08:45,138 - NorthCarolina.metamodel - INFO - Spectral site   6 / 216: 5m_15
2026-06-17 16:08:45,993 - NorthCarolina.metamodel - INFO - Spectral site   7 / 216: 5m_16
2026-06-17 16:08:46,854 - NorthCarolina.metamodel - INFO - Spectral site   8 / 216: 5m_17
2026-06-17 16:08:47,726 - NorthCarolina.metamodel - INFO - Spectral site   9 / 216: 5m_18
2026-06-17 16:08:48,585 - NorthCa

<xarray.Dataset> Size: 5MB
Dimensions:      (sites: 216, time: 744)
Coordinates:
  * sites        (sites) object 2kB '5m_10' '5m_11' ... 'buoy_8658120'
    site         (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
  * time         (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
Data variables:
    hs           (sites, time) float64 1MB 0.7368 0.7364 ... 0.5332 0.5306
    tp           (sites, time) float32 643kB 4.584 4.671 4.756 ... 10.77 10.77
    tm01         (sites, time) float64 1MB 4.078 4.182 4.294 ... 5.609 5.697
    dm           (sites, time) float64 1MB 174.1 172.7 169.9 ... 84.24 84.92
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

In [7]:
import numpy as np

spectral_filtered = spectral.where(spectral.hs.max(dim="time") < 10.0, np.nan).dropna(
    dim="sites"
)
spectral_filtered

<xarray.Dataset> Size: 4MB
Dimensions:      (sites: 209, time: 744)
Coordinates:
  * sites        (sites) object 2kB '5m_10' '5m_11' ... 'buoy_8658120'
    site         (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 8kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
  * time         (time) datetime64[ns] 6kB 1986-01-01 ... 1986-01-31T23:00:00
Data variables:
    hs           (sites, time) float64 1MB 0.7368 0.7364 ... 0.5332 0.5306
    tp           (sites, time) float32 622kB 4.584 4.671 4.756 ... 10.77 10.77
    tm01         (sites, time) float64 1MB 4.078 4.182 4.294 ... 5.609 5.697
    dm           (sites, time) float64 1MB 174.1 172.7 169.9 ... 84.24 84.92
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

## 4 · Save

In [8]:
out_path = mm.save_spectral(spectral_filtered)
print(f"wrote {out_path}")
print(f"variables: {list(spectral_filtered.data_vars)}")

2026-06-17 16:11:53,630 - NorthCarolina.metamodel - INFO - Wrote spectral metamodel to /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc (209 site(s), 4.39 MB)


wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc
variables: ['hs', 'tp', 'tm01', 'dm']
